### This code is for ensuring that the peakmatrices are formatted correctly for COGS/multiCOGS input. The PMs provided in OSF as extended data (S5, ILC and S9, CD4) are the properly formatted PMs.

In [ ]:
## Run the pre-COGS modification of the peak matrices.
## Original script was ~/HRJ_monocytes/hILCs/scripts/helen_scripts_for_rCOGS_in/ABC_thresholds_Jan2025/01_modify_extended_peakMatrices.R, now archived.

### The new peakmatrices as of Jan 2025 have slightly modified header names, and the ABCC score is also given as the raw score rather than above 5
### Problems with the PMs:
### 1. Change the name "baitID_fres" to "baitID"
### 2. Change the ABCC score to 5.1 (when not NA)
### 3. oeChr column is missing for ABC.Score, 
### 4. For many ABC interactions we are missing the "baitName" for the gene - 
        # However, this will be added when we annotate the PM for COGS using TSS locations, using COGS input scripts.
####################

### Addressing points 1-3: Function for making the PMs as input for rCOGS
source("~/Rfunctions/helen_functions.R")
library(data.table)
library(dplyr) # needed to collapse gene names in the second function.

setwd("~/HRJ_monocytes/hILCs/rCOGS_in/Version3_revision2/peakmatrices")

rmap <- fread("~/spivakov/Design/Human_hg38_DpnII_75_1200/hg38_dpnII.rmap")
names(rmap) = c("oeChr", "oeStart", "oeEnd", "oeID")
rmap_small <- rmap[, .(oeChr, oeID)]

modify_pm_for_cogs <- function(pm_location) {
    x <- fread(pm_location)
    # The bait fragment ID will be joined with the annotated baitmap to get gene promoters. In the COGS scripts, it is called "baitID".
    # The other end ID also needs to be "oeID".
    # The names for fragment coordinates also matter, because we do not expand 5kb bins for the extended PM; therefore, the frags are not added later.
    setnames(x, c("bait_start_fres", "bait_end_fres", "baitID_fres", "oe_start_fres", "oe_end_fres", "oeID_fres"), 
                c("baitStart", "baitEnd", "baitID","oeStart", "oeEnd", "oeID"))
    # The ABCC score is set to a threshold (Jan 2025, 0.23 for ILC3 and CD4+ T cells.) 
    # However, COGS works at the threshold of >5, for PCHiC. Therefore, set all ABCC scores to 5.1 
    x[!is.na(ABC.Score), ABC.Score := 5.1]
    # The oeChr column is missing for ABC.Score! Add it in, based on the fragment ID.
    x[, oeChr := NULL]
    setkey(x, "oeID")
    x_new <- rmap_small[x, on = "oeID", nomatch = NULL]
    # col. "oeChr" should have been added.
    x_new_final <- x_new[, .(baitChr, baitStart, baitEnd, baitID, baitName, oeChr, oeStart, oeEnd, oeID, baitID_5kb, oeID_5kb, dist, 
                            N_fres, N_5kb, N_abc, chicago_score_fres, chicago_score_5kb, ABC.Score)]
    return(x_new_final)
}

Make the input PMs for multiCOGS/COGs.
Note these modified input PMs will be provided on OSF as extended data files. The user can then run the wrapper scripts directly. Link: https://osf.io/aq9fb/overview

In [10]:
ILC3_extended_old <- fread("~/spivakov/miniPCHiC/hILCs/ILC3/PCHiC/data/ILC3_chicago_fres_bin_5kb_abc_023_fres_extended_peakm_13012025.txt")
#ILC3_extended_old
ILC3_extended <- modify_pm_for_cogs("~/spivakov/miniPCHiC/hILCs/ILC3/PCHiC/data/ILC3_chicago_fres_bin_5kb_abc_023_fres_extended_peakm_13012025.txt")
fwrite_headers(ILC3_extended, "./ILC3_chicago_fres_bin_5kb_abc_023_fres_extended_peakm_13012025_modified.txt")
# checked, this modified pm has the same number of lines as the original PM.
#print((ILC3_extended[!is.na(ABC.Score)]))

In [11]:
CD4_extended <- modify_pm_for_cogs("~/spivakov/miniPCHiC/PCHiC_DpnII_merged/CD4_chicago_fres_5kb_abc_023_fres_extended_peakm_13012025.txt")
fwrite_headers(CD4_extended, "./CD4_chicago_fres_5kb_abc_023_fres_extended_peakm_13012025_modified.txt")
#print(head(CD4_extended[!is.na(ABC.Score)]))